# Classical Simulation via Stabiliser State Decomposition

The initial groundwork for utilising ZX-Calculus for classical simulation via stabiliser decomposition comes from this 2021 paper by Kissinger and van de Wetering: https://arxiv.org/abs/2109.01076. Since then, there has been much literature on this topic, with new decompositions found as well as new heuristics and strategies for applying the decompositions already known. At the end of this notebook, a list of relevant publications to date is included, and for an explanation of the core concepts and review of the literature see section 2.3 (and in particular section 2.3.4) of https://ora.ox.ac.uk/objects/uuid:54d6e7f4-b845-4275-98cd-cf6749530d9b/files/d9s1616869.

This notebook aims to demonstrate the high-level API in PyZX for classically simulating circuits with stabiliser decomposition. For a deeper explanation of the specific methods available, it is advised to read the publications associated with the decompositions and strategies of interest (or to examine the relevant py file in pyzx.simulation.decompositions or pyzx.simulation.strategies), which are listed in the metadata and may be retrieved as shown later in this notebook. Moreover, this notebook primarily focuses on strong simulation (as this is more thoroughly researched and implemented with ZX-Calculus).

In [3]:
!pip install pyzx

In [4]:
import pyzx as zx
from pyzx.simulation import Decomp, Strategy

In [5]:
zx.simulation.get_reference(Decomp.CUT_EDGE)

'https://arxiv.org/pdf/2403.10964, https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/codsi-thesis.pdf'

In [35]:
zx.simulation.get_reference(Decomp.CUT_WISHBONE)

'https://arxiv.org/abs/2412.17182, https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/ahmad-thesis.pdf'

In [ ]:
zx.simulation.get_reference(Decomp.CUT_VERTEX)

'https://arxiv.org/pdf/2403.10964, https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/codsi-thesis.pdf'

In [93]:
zx.simulation.get_reference(Decomp.MAGIC_2)

'https://arxiv.org/pdf/2202.09202, https://arxiv.org/pdf/2109.01076'

In [57]:
import inspect

In [106]:
function = zx.simulation.decompositions.get_decomp(Decomp.CAT_3)

In [108]:
print(inspect.signature(function))

(g: pyzx.graph.base.BaseGraph[~VT, ~ET], v: ~VT) -> pyzx.simulation.common.SumGraph


In [6]:
import pyzx as zx
from pyzx.simulation import Decomp, Strategy

In [109]:
print(inspect.signature(function))

(g: pyzx.graph.base.BaseGraph[~VT, ~ET], v: ~VT) -> pyzx.simulation.common.SumGraph


### Applying Decompositions

The following shows a minimal example of how to apply a stabiliser state decomposition to a ZX-diagram:

In [123]:
# Generate very simple example graph consisting of two connected Z-spiders:
g = zx.Graph()

g.add_vertex(zx.VertexType.Z,0,0)

g.add_vertex(zx.VertexType.Z,2,4)

g.add_vertex(zx.VertexType.Z,0,8)

g.add_vertex(zx.VertexType.Z,6,12)

g.add_vertex(zx.VertexType.Z,8,16)

g.add_vertex(zx.VertexType.Z, 10, 20)

g.add_edge((0,1))
g.add_edge((1,2))
g.add_edge((2,3))
g.add_edge((3,4))
g.add_edge((4,5))

g.add_edge((2,0))



zx.draw(g, labels=True)


# Apply an instance of the edge cutting decomposition to edge (0,1) of graph g, with vertex type X:
'''terms = zx.simulation.apply_decomp(Decomp.CUT_EDGE, g=g, e=(0,1), ty=zx.VertexType.Z)'''

#Apply an instance of the edge vertex decomposition to vertex 1
'''terms = zx.simulation.apply_decomp(Decomp.CUT_VERTEX, g=g, v= 1)'''

#Apply an instance of the edge cut wishbone decomposition to vertex 0
'''terms = zx.simulation.apply_decomp(Decomp.CUT_WISHBONE, g=g, v=2)'''

#Apply an instance of the magic_2 decomposition to vertex 0 and 1
'''terms = zx.simulation.apply_decomp(Decomp.MAGIC_2, g=g, verts=[0,2])'''


#Apply an instance of the magic_5 decomposition to vertex 0,1,2,3,4
'''terms = zx.simulation.apply_decomp(Decomp.MAGIC_5, g=g, verts=[0,1,2,3,4])'''

#Apply an instance of the cat_3 decomposition to vertex 2
'''terms = zx.simulation.apply_decomp(Decomp.CAT_N, g=g, v=2)'''

#Apply an instance of BSS on our 6 Z spiders
terms = zx.simulation.apply_decomp(Decomp.BSS, g=g, verts=[0,1,2,3,4,5])

# Draw the two branching terms:
for term in terms.graphs:
    zx.draw(term)

In [ ]:
help(zx.simulation.apply_decomp)

Help on function apply_decomp in module pyzx.simulation.decompositions:

apply_decomp(kind: Decomp, g: BaseGraph[VT, ET], *args, **kwargs) -> SumGraph
    Applies an instance of the specified decomposition to the provided graph with the decomposition-specific arguments.



My own example to understand playing around

In [ ]:
''' cut edge is a rule you need to learn and apply
The definition is the following :  ----- = ---o     o---    +   ---o π   π  o---
    the best way I found for now is the following 
    Suppose at the beginning you have   
    A------------B------------C 
    And that you want to cut the edge between A and B
    The best way to perform the cut edge is to do :
    1) writting twice the diagram without the connexion
    A            B------------C
                                  
    A            B------------C
    2) on the first diagram, apply  ---o        o---
       on the second diagram, apply ---o π   π  o---
    A---o      o----B---------C
    A---o π  π o----B---------C
    3) the final result is just the sum of both 
     A---o      o----B---------C  + 
     A---o π  π o----B---------C

CUT_EDGE  apply_decomp(Decomp.CUT_EDGE, g, e=(A,B), ty=Z)
    Cut one wire. A and B stay. Two new spiders of color ty (default Z) are not wired to each other.
    A--------B--------C
    = A---o    o---B--------C
    + A---oπ  πo---B--------C


CUT_VERTEX  apply_decomp(Decomp.CUT_VERTEX, g, v=V)
    Delete V. On every leg that touched V, put one spider of the opposite color. Those new spiders are not wired to each other.
    A--------V--------B          V was green, so the new spiders are red
    = A---o         o---B
    + A---oπ       πo---B
    A spider with one leg only looks like a color swap: A---V becomes A---o, plus A---oπ.

 CUT_WISHBONE  apply_decomp(Decomp.CUT_WISHBONE, g, v=V, neighs=[C], ph=0)
    Split V into two spiders of the same color, then cut the wire between them.
    neighs go to the right spider. Every other neighbor stays on the left.
    Right phase = ph. Left phase = old phase of V minus ph.
    The two new spiders are not wired to each other. The second term adds π to both.
    A----V----B                neighs=[B], ph=0
         |
         C
    = A----L          R----B
                      |
                      C
    + A----Lπ        πR----B
                      |
                      C
    neighs=[] keeps every old wire on L, and R is a ball with no wire.


 MAGIC_2  apply_decomp(Decomp.MAGIC_2, g, verts=[P,Q])
    Two Z spiders stay. Each loses π/4. A new Z is tied to both. Two terms.
    P        Q
    = P--o(π/2)--Q          plain wires
    + P~~o(π)~~Q            Hadamard wires, and the scalar gains phase π/4


MAGIC_5  apply_decomp(Decomp.MAGIC_5, g, verts=[T1,T2,T3,T4,T5])
    Five spiders stay. They are not deleted. Three terms.
    In every term, each of the five loses π/4 and is wired to a new hub H, and H has a one-leg appendix A.
    The three terms differ by the phases on the five spiders, on H and on A, and by plain wires versus Hadamard wires.
    T1 T2 T3 T4 T5
    = all five wired to H---A
    + all five wired to H---A     other phases, Hadamard
    + all five wired to H---A     other phases, Hadamard, minus sign


CAT_3  apply_decomp(Decomp.CAT_3, g, v=K)
    K must have degree 3 and phase 0 or π. K is deleted. Its three neighbors stay and each loses π/4. Two terms.
    Each term hangs those three neighbors on a new hub H with a one-leg appendix A.
    T1 T2 T3 all wired to K
    = T1 T2 T3 wired to H---A
    + T1 T2 T3 wired to H---A     other phases; some wires become Hadamard


CAT_4  apply_decomp(Decomp.CAT_4, g, v=K)
    Same rule as CAT_3, with 4 neighbors and 2 terms.


CAT_5  apply_decomp(Decomp.CAT_5, g, v=K)
    Same rule as CAT_3, with 5 neighbors and 3 terms.


CAT_6  apply_decomp(Decomp.CAT_6, g, v=K)
    Same rule as CAT_3, with 6 neighbors and 3 terms.


CAT_N  apply_decomp(Decomp.CAT_N, g, v=K)
    Reads the degree of K and calls CAT_3, CAT_4, CAT_5, or CAT_6. Degree must be 3, 4, 5, or 6. Phase of K must be 0 or π.


BSS  apply_decomp(Decomp.BSS, g, verts=[Z1,Z2,Z3,Z4,Z5,Z6])
    Exactly 6 Z spiders. They stay. Seven terms. In every term each of the six loses π/4.
    The seven terms then differ:
      1. phase change only
      2. an extra π on each of the six
      3. an extra π/2 on each, plus a new Z of phase π wired to all six by Hadamard
      4. an extra π/2 on each, plus a new Z of phase 0 wired to all six by Hadamard
      5. a new Z of phase 3π/2 wired to all six by plain wires
      6. a small extra gadget on the first five; the sixth gets a phase-π stub
      7. the same gadget with spiders 3 and 6 swapped
    Each term also carries its own scalar. The order of the six spiders changes terms 6 and 7.

'''
    
    

#create a graph with nothing
g2 = zx.Graph()
#add a Z spider
g2.add_vertex(zx.VertexType.X,0,0,0)
#add an X spider
g2.add_vertex(zx.VertexType.X,0,4,0)
#add a Z gate, it is just a Z spider with phase pi
g2.add_vertex(zx.VertexType.X,0,8,1)
#draw an edge between the Z spider and the X spider
g2.add_edge((0,1))
#draw an edge between the X spider and the Z gate
g2.add_edge((1,2))
#draw the initial circuit
zx.draw(g2,labels=True)
#apply the cut edge between the Z spider and the X spider
term = zx.simulation.apply_decomp(Decomp.CUT_EDGE, g=g2, e=(0,1))
#draw the first new term
zx.draw(term.graphs[0],labels=True)
#draw the second new term
zx.draw(term.graphs[1],labels=True)




In [11]:
help(g2.add_vertex)

Help on method add_vertex in module pyzx.graph.base:

add_vertex(
    ty: VertexType = <VertexType.BOUNDARY: 0>,
    qubit: FloatInt = -1,
    row: FloatInt = -1,
    phase: FractionLike | None = None,
    ground: bool = False,
    index: VT | None = None
) -> VT method of pyzx.graph.graph_s.GraphS instance
    Add a single vertex to the graph and return its index.
    The optional parameters allow you to respectively set
    the type, qubit index, row index and phase of the vertex.
    For H-boxes and Z-boxes with complex labels, use set_h_box_label
    or set_z_box_label after creating the vertex.



Another example, with a larger graph, follows. In this case, we apply the magic2 decomposition which, instead of a graph and an edge, takes a graph and a list of vertices as arguments.

In [12]:
# Generate a larger scalar Clifford+T ZX-diagram:
q = 10
d = 350
seed = 42
g = zx.generate.cliffordT(q,d,seed=seed)
g.apply_state("0"*q)
g.apply_effect("0"*q)
zx.draw(g,scale=20,labels=True)
zx.simplify.full_reduce(g)
g.normalize()
zx.draw(g,scale=20,labels=True)

My own example on cliffordT, rerun multiple time the cell you'll see the difference
the first parameter is the number of qubits = the number of horizontal lines you have
the second parameter is the depth = the number of pi over two gates max you ll gate, how many max green,red spider with a phase or without if it s a control gate

In [13]:
help(zx.full_reduce)

Help on function full_reduce in module pyzx.simplify:

full_reduce(
    g: BaseGraph[VT, ET],
    matchf: Optional[Callable[[Union[VT, ET]], bool]] = None,
    quiet: bool = True,
    stats: Optional[Stats] = None
) -> None
    The main simplification routine of PyZX. It uses a combination of :func:`clifford_simp` and
    the gadgetization strategies :func:`pivot_gadget_simp` and :func:`gadget_simp`. It also attempts to run :func:`supplementarity_simp` and :func:`copy_simp`.



why the full reduce does not work after apply_state and apply_effect... or work to well ?

In [14]:
help( zx.generate.cliffordT(q,d).normalize)

Help on method normalize in module pyzx.graph.base:

normalize() -> None method of pyzx.graph.graph_s.GraphS instance
    Puts every node connecting to an input/output at the correct qubit index and row.



In [15]:
q = 10
d = 350
seed =30
#generate a clifford diagram with 5 qubits, max 10T, and with a seed 30
g = zx.generate.cliffordT(q,d, seed=seed)
zx.draw(g)

#using the full reduce rule to simplify the circuit it uses graph from ans some simplification
#zx.simplify.full_reduce(g)
#zx.draw(g)

#put some red spiders without phase at the beginning and at the end of the circuit
g.apply_state("0"*q)
g.apply_effect("0"*q)
zx.draw(g)



#applying full reduce again
zx.simplify.full_reduce(g)
zx.draw(g)

#check what the normalization do
g.normalize()
zx.draw(g)



Another example, with a larger graph, follows. In this case, we apply the magic2 decomposition which, instead of a graph and an edge, takes a graph and a list of vertices as arguments.

In [16]:
# Generate a larger scalar Clifford+T ZX-diagram:
q = 10
d = 350
seed = 42
g = zx.generate.cliffordT(q,d,seed=seed)
g.apply_state("0"*q)
g.apply_effect("0"*q)
zx.draw(g,scale=20,labels=True)
zx.simplify.full_reduce(g)
#g.normalize()
zx.draw(g,scale=20,labels=True)

In [17]:
# apply the magic2 decomposition to vertices 479 and 506 of graph g:
#terms = zx.simulation.apply_decomp(Decomp.MAGIC_2,g,[44,51])
#terms = zx.simulation.apply_decomp(Decomp.CUT_EDGE,g=g,e=(44,54))
#terms = zx.simulation.apply_decomp(Decomp.CUT_EDGE, g=g, e=(30,37))
terms = zx.simulation.apply_decomp(Decomp.MAGIC_5,g,[479,56,67,479,96])

# display both branching terms:
for h in terms.graphs:
    zx.draw(h,scale=20,labels=True)

Or if we don't want to actually apply the decomposition here but just check if it would be a valid application, we can run:

In [18]:
zx.simulation.check_valid(Decomp.MAGIC_5,g,[479,56,67,479,96])

True

This validity check is also automatically run before the decomposition is applied when zx.simulation.apply_decomp(...) is called.

As with zx.simulation.apply_decomp, the specific function this runs under the hood is tied to the specific decomposition but should always return True if the specified application of the decomposition is valid. If instead the specified application is not valid, it should throw an appropriate error like so:

In [19]:
zx.simulation.check_valid(Decomp.MAGIC_2,g,[479,506,227])

AssertionError: Invalid application of magic2 decomposition. Expected 2 vertices but 3 were specified.

Both zx.simulation.apply_decomp and zx.simulation.check_valid may take the same arguments for a particular decomposition but these arguments will vary from one decomposition to another. The first argument is always the Decomp enum entry (of type zx.simulation.Decomp) and the second argument is always the graph to which the decomposition is to be applied. Most decompositions will also require one or more subsequent arguments, such as an edge or vertex or list of vertices, etc. If in doubt, check the specification of the decomposition you're using from its file under pyzx/simulation/decompositions.

### (Preparation for) Simulating Circuits

Circuits can be created or imported into PyZX in a number of ways, such as via QASM code with zx.Circuit.load("circuit.qasm") or directly in code:

In [ ]:
qasm = f"""
OPENQASM 2.0;
include "qelib1.inc";

qreg q[3];
h q[0];
x q[0];
z q[0];
h q[1];
cx q[0], q[1];
t q[1];
t q[2]
"""

circ = zx.Circuit.from_qasm(qasm)
zx.draw(circ)


Alternatively, various types of random circuits can be generated with the pyzx.generate module, such as follows:

In [ ]:
# Generate a scalar Clifford+T ZX-diagram:
q = 10    # number of qubits
d = 250   # number of gates (~depth)
seed = 42 # seed for random generation
g = zx.generate.cliffordT(q,d,seed=seed)
zx.draw(g,scale=20)

To strongly simulate a circuit with stabiliser decomposition in PyZX, it must first be a *scalar* ZX-diagram, meaning it must have no open inputs or outputs (i.e. no boundary vertices). If the ZX-diagram is circuit-like then the inputs and outputs can be plugged with states and effects like so:

In [ ]:
bitstr = "1111011111" # some output bitstring whose probability we want to determine for this graph
g.apply_state("0"*q)
g.apply_effect(bitstr)
zx.draw(g,scale=20,labels=True)

Here, bitstr="0011010101" is a particular output bitstring we want to check. Scroll to the right end of the graph and you'll see the 0's correspond to 0-phase X-spiders and the 1's pi-phase X-spiders. By plugging the output of the circuit in this way, we will be aiming to ultimately calculate *"what is the probability that this quantum circuit will produce the output 0011010101 if it were to be executed on a (noiseless) quantum computer?"* This is the essence of strong classical simulation.

Next, we simplify the graph with the ZX rewriting rules via full_reduce:

In [ ]:
zx.simplify.full_reduce(g)
zx.draw(g,scale=20,labels=True)
g.normalize()
zx.draw(g,scale=20,labels=True)

Note that this (and all) ZX-diagrams also have an associated complex scalar factor not displayed, which may be exposed like so:

In [ ]:
print(g.scalar)

32.00+0.00i = sqrt(2)^10


Or expressed as a complex number rather than the more structured scalar data structure of PyZX:

In [20]:
print(g.scalar.to_number())

(-94906265.6242519+1.1622665441576857e-08j)


In the case of a *Clifford* scalar ZX-diagram, full_reduce will contract the entire graph to nothing more than a complex scalar. But in general, when the circuit includes non-Clifford gates (spiders of phases beyond n*pi/2 for integer n) the rewriting rules alone are insufficient to fully reduce the graph to a scalar, as observed in the example above. It will nevertheless typically still reudce the number of non-Clifford gates to some extent, which is very helpful for classical simulation (see: https://arxiv.org/abs/2109.01076).

### Simulating Circuits

A non-Clifford scalar ZX-diagram can then be reduced to its scalar amplitude with the help of stabiliser decomposition:

In [21]:
%%time
# draw the post-full_reduce example graph from earlier:
zx.draw(g,scale=20)

# fully decompose the graph using the 'BSS' decomposition strategy of https://arxiv.org/abs/2109.01076:
terms = zx.simulation.full_decompose(Strategy.BSS,g)

CPU times: user 20.9 s, sys: 30.2 ms, total: 20.9 s
Wall time: 21.1 s


Here, we selected the 'BSS' decomposition strategy (https://arxiv.org/abs/2109.01076) to fully decompose the graph into a list (assigned to the 'terms' variable) of empty scalar graphs (essentially just complex scalar factors) whose sum represents the amplitude of the original graph.

We can now sum these terms together to determine the amplitude:

In [22]:
sum(term.scalar.to_number() for term in terms) # calculate the amplitude by summing the scalar terms

(-0.02984018475205449+0.01556155160404748j)

In [23]:
count = 0
for term in terms : 
    
    print (term.scalar.to_number())
    count += 1
print (count)

(-1.2695904826402922e-19-0.002073398605253744j)
(0.0006072843913760924+0.001466114213877652j)
(0.00010419352912516165+0.00010419352912516158j)
(0.000355738960250643-0.0003557389602506431j)
(0.0017176596450031174+0.002429137565504363j)
(-0.0024291375655043725+3.81397057270388e-19j)
(0.0020733986052537483+0.0008588298225015615j)
(0.0003557389602506251+0.002073398605253748j)
(-0.000858829822501558+0.0008588298225015586j)
(0.001214568782752182+0.0012145687827521815j)
(-0.0008588298225015583-0.0008588298225015582j)
(-0.00010419352912516169-0.0002515454311254813j)
(0.00010419352912516723+4.3158372875160316e-05j)
(0.00020838705825032281-0.0005030908622509619j)
(-0.0005894076080012801+7.218161405318655e-20j)
(-0.00020838705825032357+0.00020838705825032374j)
(-6.103515625000332e-05+0.0003557389602506422j)
(-5.4136210539889706e-20-0.0002947038040006389j)
(-0.00029470380400063874+0.00020838705825032295j)
(-0.0012145687827521815-0.0017176596450031154j)
(-6.10351562500036e-05-0.0003557389602506424j

Or we can fully decompose and return the scalar directly by using *pyzx.simulation.simulate* instead of *pyzx.simulation.full_decompose*:

In [24]:
%%time
# fully decompose the graph using the 'BSS' decomposition strategy and sum the resulting terms to determine the overall amplitude:
amplitude = zx.simulation.simulate(Strategy.BSS,g)
print(amplitude)

(-0.029840184752054733+0.015561551604047569j)
CPU times: user 23.8 s, sys: 38.5 ms, total: 23.8 s
Wall time: 24 s


In [25]:
amplitude = zx.simulation.simulate(Strategy.CUT_RANDOM,g)
print(amplitude)

(-0.029840184752054737+0.01556155160404726j)


Thus we have deduced the scalar amplitude that the plugged ZX-diagram g represents. Relating back to the initial circuit, this then tells us the probability amplitude *A* of this circuit given the inputs (|00...0>) and outputs (<0011010101|>) we specified.

As such, we can now answer our initial question of *"what is the probability that this quantum circuit will produce the output 0011010101 if it were to be executed on a (noiseless) quantum computer?"*, and thus achieve strong classical simulation, by taking the absolute square of this amplitude:

In [26]:
# determine the probability of measuring this particular output bitstring (0011010101) from this circuit:
prob = abs(amplitude)**2
print(prob)

0.001132598514362186


And just like that, we've strongly classically simulated our circuit.

Moreover, given we know the initial T-count of the graph (after full_reduce) as well as the number of terms produced by full_decompose, we can quantify the *effective alpha* (see later section) of this decomposition strategy (at least for this particular graph):

In [27]:
import math

t = zx.tcount(g)           # get the T-count of graph g
n = len(terms)             # count the number of terms produced after fully decomposing with the BSS strategy
alpha_eff = math.log2(n)/t # Calculate the effective alpha for this strategy (as applied to this graph)

print("Initial T-count:",t)
print("#terms produced:",n)
print("Effective alpha:",alpha_eff)

Initial T-count: 30
#terms produced: 5273
Effective alpha: 0.41214694271266106


Thus we see that by using ZX-simplification after every application of a decomposition, we achieve more efficient simulation scaling than would be achieved otherwise, as this measured effective alpha is lower than the theoretical ("naive") alpha of this decomposition:

In [28]:
zx.simulation.get_alpha(Decomp.BSS)

0.4678924870096

In [29]:
list[Decomp] (Decomp)

[<Decomp.BSS: 'bss'>,
 <Decomp.CAT_3: 'cat_3'>,
 <Decomp.CAT_4: 'cat_4'>,
 <Decomp.CAT_5: 'cat_5'>,
 <Decomp.CAT_6: 'cat_6'>,
 <Decomp.CAT_N: 'cat_n'>,
 <Decomp.MAGIC_5: 'magic_5'>,
 <Decomp.MAGIC_2: 'magic_2'>,
 <Decomp.CUT_EDGE: 'cut_edge'>,
 <Decomp.CUT_VERTEX: 'cut_vertex'>,
 <Decomp.CUT_WISHBONE: 'cut_wishbone'>]

### Additional Tips and Tricks

The decompositions also have associated metadata which may be useful to the user. For instance, the naive 'alpha' coefficient of a decomposition quantifies how efficient it is at removing T-spiders. If no ZX-rewriting simplification is applied during decomposition and all t T-spiders in a graph are removed entirely by recursive applications of a particular decomposition, the number of final terms produced would be 2^{alpha*t}. This is essentially how alpha is defined for a decomposition, namely alpha:=log_2(num_terms)/(T-count). (See definition 17 of https://ora.ox.ac.uk/objects/uuid:54d6e7f4-b845-4275-98cd-cf6749530d9b/files/d9s1616869.)

As such, a lower alpha decomposition is better (at least naively). (In practice, applying higher alpha decompositions with thoughtful heuristics can lead to fewer terms overall, after ZX-simplification is taken into account, resulting in a lower "effective alpha". See: https://arxiv.org/abs/2403.10964)

In PyZX, as demonstrated above, the naive alpha coefficient of a decomposition can be retrieved like so:

In [ ]:
zx.simulation.get_alpha(Decomp.MAGIC_5)

0.3962406251803

For most decompositions, relevant literature references can also be retrieved like so:

In [ ]:
zx.simulation.get_reference(Decomp.CUT_VERTEX)

'https://arxiv.org/pdf/2403.10964, https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/codsi-thesis.pdf'

This also works for decomposition *strategies*:

In [ ]:
zx.simulation.get_reference(Strategy.BSS)

'https://arxiv.org/abs/2109.01076'

The decompositions and strategies currently available in PyZX can be found under the pyzx.simulation.decompositions and pyzx.simulation.strategies folders respectively, or with IntelliSense on the pyzx.simulation.Decomp and pyzx.simulation.Strategy enums. Alternatively, the options may be listed thusly:

In [ ]:
list(Decomp)

[<Decomp.BSS: 'bss'>,
 <Decomp.CAT_3: 'cat_3'>,
 <Decomp.CAT_4: 'cat_4'>,
 <Decomp.CAT_5: 'cat_5'>,
 <Decomp.CAT_6: 'cat_6'>,
 <Decomp.CAT_N: 'cat_n'>,
 <Decomp.MAGIC_5: 'magic_5'>,
 <Decomp.MAGIC_2: 'magic_2'>,
 <Decomp.CUT_EDGE: 'cut_edge'>,
 <Decomp.CUT_VERTEX: 'cut_vertex'>,
 <Decomp.CUT_WISHBONE: 'cut_wishbone'>]

In [ ]:
list(Strategy)

[<Strategy.BSS: 'bss'>,
 <Strategy.CUT_RANDOM: 'cut_random'>,
 <Strategy.MAGIC_CAT: 'magic_cat'>]

### Related Literature / Further Reading

1.    **"Trading classical and quantum computational resources"** (https://arxiv.org/abs/1506.01396) - BSS decomp. (6 T-gates -> 7 Clifford terms: alpha~=0.47)

2.    **"Simulating quantum circuits with ZX-calculus reduced stabiliser decompositions"** (https://arxiv.org/abs/2109.01076) - The initial paper on utilising ZX for stab decomp classical sim. Specifically, translating BSS into ZX and applying full_reduce after each decomp. to further reduce T-count

3.    **"Classical simulation of quantum circuits with partial and graphical stabiliser decompositions"** (https://arxiv.org/abs/2202.09202) - Cats decomps (lower alpha, but structure specific) + Magic5 decomp (state of the art alpha~=0.396)

4.    **"Cutting-Edge Graphical Stabiliser Decompositions for Classical Simulation of Quantum Circuits"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/codsi-thesis.pdf) - Early cutting decomp work (chapters 3-4), plus some gate-by-gate weak simulation (chapter 8)

5.    **"Speedy Contraction of ZX Diagrams with Triangles via Stabiliser Decompositions"** (https://arxiv.org/abs/2307.01803) - Finding new decomps. with triangle and star operators

6.    **"Graphical Stabilizer Decompositions For Counting Problems"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/laakkonen-thesis.pdf) - Similar to above, finding new decomps. with H-boxes

7.    **"Classically Simulating Quantum Supremacy IQP Circuits through a Random Graph Approach"** (https://arxiv.org/abs/2212.08609) - Extending ideas from [4] and applying them to IQP circuits 

8.    **"Procedurally Optimised ZX-Diagram Cutting for Efficient T-Decomposition in Classical Simulation"** (https://arxiv.org/abs/2403.10964) - Using the cutting decomp from [4], demonstrating how analysing the circuit in advance to find good cuts can lead to more efficient term counts than using a lower alpha decomposition. (Foundational to the KH papers [19,20])

9.   **"Fast Classical Simulation of Quantum Circuits via Parametric Rewriting in the ZX-Calculus"** (https://arxiv.org/abs/2403.06777) - "ParamZX" (precursor to TSim [21]), showing that the rewriting rules can (fully for Cliffords and almost fully for Clifford+T) be preserved for (appropriately restricted) parameterised phases such that branching terms can be expressed as a single parameterised diagram, leading to a parametric scalar ideal for efficient GPU computation. Average speedup factor of 78x.

10.   **"Smarter k-Partitioning of ZX-Diagrams for Improved Quantum Circuit Simulation"** (https://arxiv.org/abs/2409.00828) - a hyrbid method combining stabiliser decomposition with tensor contraction via graph partitioning

11.   **"Efficient Heuristics for Classical Simulation of Quantum Circuits Using ZX-Calculus"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/ahmad-thesis.pdf) - Building off of [8], showing some new (low alpha but very structure specific and situational) decompositions derived from special cases of the cutting decomp.

12.   **"Dynamic T-decomposition for classical simulation of quantum circuits"** (https://arxiv.org/abs/2412.17182) - Paper from [11] MSc thesis

13.   **"Towards Faster Quantum Circuit Simulation Using Graph Decompositions, GNNs and Reinforcement Learning"** (https://openreview.net/pdf?id=54060pbCKY) - Using reinforcement learning rather than random choice when selecting groups of spiders with which to apply Magic5 achieves marginally better results.

14.   **"GNNs for Fast Classical Simulation of Quantum Circuits through Optimizing ZX-Graph Decompositions"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/philipps-thesis.pdf) - related to [13], an MSc thesis on how ML can be used to help optimise the decomp. routine

15.   **"Novel methods for classical simulation of quantum circuits via ZX-calculus"** (https://ora.ox.ac.uk/objects/uuid:54d6e7f4-b845-4275-98cd-cf6749530d9b/files/d9s1616869) - PhD thesis covering papers [8,9,10,12,13]

16.   **"A Rank-Width–Based Approach to Quantum Circuit Simulation via ZX-Calculus"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/kuyanov-thesis.pdf) - A rank-width approach, using ZX for a tensor contraction approach to simulation

17.   **"Efficient Classical Simulation of Low-Rank-Width Quantum Circuits Using ZX-Calculus"** (https://arxiv.org/pdf/2603.06764) - A paper based on the [16] MSc thesis

18.   **"Unifying Graph Measures and Stabilizer Decompositions for the Classical Simulation of Quantum Circuits"** (https://arxiv.org/pdf/2603.06377) - Another hybrid approach based on tensor contraction and stabiliser decomposition

19.   **"Cutting stabiliser decompositions of magic state cultivation with ZX-calculus"** (https://arxiv.org/abs/2509.01224) - Cutting decomp techniques realted to [8] applied to magic state cultivation circuits, resulting in hugely reduced term counts

20.   **"Simulating magic state cultivation with few Clifford terms"** (https://arxiv.org/abs/2509.08658) - Followup paper to [19]

21.   **"Tsim: Fast Universal Simulator for Quantum Error Correction"** (https://arxiv.org/abs/2604.01059) - "TSim" open source software applying the parameterised/GPU-parallelised work of [9] to quantum error correction

<sup><sub>Jupyter Notebook by @mjsutcliffe99</sub></sup>